## 1. Install Required Packages

In [ ]:
import subprocess
import sys

# Install required packages
packages = [
    'numpy',
    'scipy',
    'nibabel',
    'torch',
    'torchvision',
    'matplotlib',
    'scikit-image',
    'scikit-learn',
    'Pillow',
    'tqdm',
    'SimpleITK'
]
,
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
,
All packages installed successfully!")

IndentationError: unexpected indent (2437799284.py, line 19)

In [ ]:
!pip install numpy spicy nibabel torch torchvision matplotlib scikit-image scikit-learn Pillow tqdm SimpleITK

## 2. Import Libraries and Custom Modules

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from pathlib import Path
import sys

# Add current directory to path for imports
sys.path.append(str(Path.cwd()))

# Import custom modules
from data_loader import MHDDataLoader
from preprocessor import PreProcessor
from model import UNet3D, CombinedLoss
from trainer import Trainer, LiverDataset
from visualizer import Visualizer

print("All modules imported successfully!")

## 3. Load and Explore Dataset

In [ ]:
# Initialize data loader
data_root = Path.cwd()
loader = MHDDataLoader(str(data_root))

# Load training data
print("Loading training data...")
train_pairs = loader.get_training_pairs()
print(f"Loaded {len(train_pairs)} training pairs")

# Load test data
print("\nLoading test data...")
test_scans = loader.get_test_scans()
print(f"Loaded {len(test_scans)} test scans")

## 4. Dataset Statistics and Analysis

In [ ]:
# Extract scans and labels
scans = [pair[0] for pair in train_pairs]
labels = [pair[1] for pair in train_pairs]
,
,
Dataset Statistics:")
print("-" * 50)
for i, (scan, label) in enumerate(train_pairs[:3]):
    liver_volume = np.sum(label)
    liver_percentage = liver_volume / label.size * 100
    print(f"Sample {i+1}:")
    print(f"  Shape: {scan.shape}")
    print(f"  Intensity range: [{scan.min()}, {scan.max()}]")
    print(f"  Liver pixels: {liver_volume} ({liver_percentage:.2f}%)")
    print()

## 5. Visualize Sample Data

In [ ]:
# Visualize first sample
fig = Visualizer.plot_multiple_slices(scans[0], labels[0], num_slices=6, axis=2)
plt.suptitle('Training Sample 1 - Multiple Slices', fontsize=14, fontweight='bold')
plt.show()

# Visualize statistics
fig = Visualizer.plot_statistics(scans[:5], labels[:5])
plt.suptitle('Dataset Statistics', fontsize=14, fontweight='bold')
plt.show()

## 6. Preprocess Data

In [ ]:
# Initialize preprocessor
preprocessor = PreProcessor(target_size=(128, 128, 64))

print("Preprocessing training data...")
processed_scans = []
processed_labels = []

for i, (scan, label) in enumerate(train_pairs):
    # Skip if too large
    if scan.size > 100_000_000:
        print(f"Skipping sample {i+1} - too large")
        continue

    try:
        # Normalize
        scan_norm = preprocessor.normalize(scan)

        # Resize
        scan_resized = preprocessor.resize(scan_norm, preprocessor.target_size)
        label_resized = preprocessor.resize_label(label.astype(np.float32), preprocessor.target_size)

        processed_scans.append(scan_resized)
        processed_labels.append(label_resized)

        print(f"✓ Processed sample {i+1}/{len(train_pairs)}")
    except Exception as e:
        print(f"✗ Error processing sample {i+1}: {e}")
        continue

print(f"\nSuccessfully processed {len(processed_scans)} samples")

## 7. Create Data Loaders

In [ ]:
# Convert to numpy arrays
X = np.array(processed_scans)
y = np.array(processed_labels)

print(f"Dataset shape: {X.shape}, Labels shape: {y.shape}")

# Create train-val split
train_size = int(0.8 * len(X))
X_train, X_val = X[:train_size], X[train_size:]
y_train, y_val = y[:train_size], y[train_size:]

print(f"Train: {X_train.shape}, Val: {X_val.shape}")

# Create datasets
train_dataset = LiverDataset(X_train, y_train, augment=True)
val_dataset = LiverDataset(X_val, y_val, augment=False)

# Create loaders
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=4, shuffle=False, num_workers=0)

print("Data loaders created!")

NameError: name 'np' is not defined

## 8. Initialize Model

In [ ]:
# Create model
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = UNet3D(in_channels=1, out_channels=1, features=32)
print(f"Model created with {sum(p.numel() for p in model.parameters()):,} parameters")

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 9. Train Model

In [ ]:
# Initialize trainer
trainer = Trainer(model, device=device)

# Setup training
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
criterion = CombinedLoss(smooth=1.0)

# Train model
print("Starting training...")
trainer.fit(
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    criterion=criterion,
    epochs=20,
    patience=5
)

print("\nTraining completed!")

## 10. Plot Training History

In [ ]:
# Plot training history
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Loss
axes[0].plot(trainer.history['train_loss'], label='Train Loss')
axes[0].plot(trainer.history['val_loss'], label='Val Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Loss Over Time')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Dice
axes[1].plot(trainer.history['val_dice'])
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Dice Score')
axes[1].set_title('Validation Dice Over Time')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Best validation loss: {min(trainer.history['val_loss']):.4f}")
print(f"Best validation dice: {max(trainer.history['val_dice']):.4f}")

## 11. Make Predictions

In [ ]:
# Get validation sample
val_sample_idx = 0
scan_tensor = torch.from_numpy(X_val[val_sample_idx:val_sample_idx+1]).unsqueeze(1).float()
label_tensor = y_val[val_sample_idx]

# Make prediction
prediction = trainer.predict(scan_tensor)
prediction = prediction.squeeze().numpy()

# Binarize prediction
prediction_binary = (prediction > 0.5).astype(np.uint8)

print(f"Prediction shape: {prediction.shape}")
print(f"Prediction value range: [{prediction.min():.3f}, {prediction.max():.3f}]")
print(f"Predicted liver pixels: {prediction_binary.sum()}")

## 12. Visualize Predictions

In [ ]:
# Compare prediction with ground truth
scan_orig = X_val[val_sample_idx]
fig = Visualizer.compare_predictions(scan_orig, label_tensor, prediction_binary, slice_idx=32)
plt.suptitle('Ground Truth vs Prediction', fontsize=14, fontweight='bold')
plt.show()

## 13. Save Model

In [ ]:
# Save model
model_path = 'liver_segmentation_model.pth'
trainer.save_model(model_path)
print(f"Model saved to {model_path}")

## 14. Evaluation Metrics

In [ ]:
from sklearn.metrics import jaccard_score, precision_score, recall_score, f1_score

# Flatten arrays
label_flat = label_tensor.flatten()
pred_flat = prediction_binary.flatten()

# Calculate metrics
jaccard = jaccard_score(label_flat, pred_flat)
precision = precision_score(label_flat, pred_flat, zero_division=0)
recall = recall_score(label_flat, pred_flat, zero_division=0)
f1 = f1_score(label_flat, pred_flat, zero_division=0)
dice = 2 * (precision * recall) / (precision + recall + 1e-7)

print("\nSegmentation Metrics:")
print("-" * 40)
print(f"Dice Score: {dice:.4f}")
print(f"Jaccard Index: {jaccard:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

## 15. Generate Predictions on Test Set

In [ ]:
# Preprocess test scans
print("Preprocessing test scans...")
test_processed = []

for i, scan in enumerate(test_scans[:3]):  # Process first 3 for demo
    try:
        scan_norm = preprocessor.normalize(scan)
        scan_resized = preprocessor.resize(scan_norm, preprocessor.target_size)
        test_processed.append(scan_resized)
        print(f"✓ Processed test sample {i+1}")
    except Exception as e:
        print(f"✗ Error: {e}")

# Make predictions
print("\nGenerating predictions...")
test_predictions = []

for i, scan in enumerate(test_processed):
    scan_tensor = torch.from_numpy(scan).unsqueeze(0).unsqueeze(0).float()
    pred = trainer.predict(scan_tensor)
    pred_binary = (pred.squeeze().numpy() > 0.5).astype(np.uint8)
    test_predictions.append(pred_binary)
    print(f"✓ Prediction {i+1} generated")

print(f"\nGenerated {len(test_predictions)} predictions")

## 16. Summary

In [ ]:
print("\n" + "="*50)
print("SLIVER07 LIVER SEGMENTATION PIPELINE SUMMARY")
print("="*50)
print(f"\nDataset:")
print(f"  - Training pairs: {len(train_pairs)}")
print(f"  - Test scans: {len(test_scans)}")
print(f"\nProcessing:")
print(f"  - Successfully processed: {len(processed_scans)} samples")
print(f"  - Target size: {preprocessor.target_size}")
print(f"\nModel:")
print(f"  - Architecture: 3D U-Net")
print(f"  - Parameters: {trainable_params:,}")
print(f"  - Device: {device}")
print(f"\nTraining:")
print(f"  - Epochs completed: {len(trainer.history['train_loss'])}")
print(f"  - Final training loss: {trainer.history['train_loss'][-1]:.4f}")
print(f"  - Final validation loss: {trainer.history['val_loss'][-1]:.4f}")
print(f"  - Best validation Dice: {max(trainer.history['val_dice']):.4f}")
print(f"\nValidation Metrics:")
print(f"  - Dice Score: {dice:.4f}")
print(f"  - Jaccard Index: {jaccard:.4f}")
print(f"  - Precision: {precision:.4f}")
print(f"  - Recall: {recall:.4f}")
print(f"\nOutput:")
print(f"  - Model saved: {model_path}")
print(f"  - Test predictions: {len(test_predictions)}")
print("\n" + "="*50)